In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/rag-from-scratch/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 00 · Setup & the corpus

RAG has exactly two moving parts: **retrieve** relevant text, then **generate**
an answer conditioned on it. This repo teaches the retrieval half by building it
from scratch, because that is where almost all RAG failures live.

**What you need**

```
pip install -r requirements.txt      # numpy, scikit-learn, sentence-transformers
```

The first embedding call downloads a ~90 MB model (`all-MiniLM-L6-v2`) and then
runs offline on CPU. **Generation** is optional: set `ANTHROPIC_API_KEY` or
`OPENAI_API_KEY` to use a real model, otherwise a deterministic *extractive*
fallback answers from the retrieved text so every notebook still runs.


In [ ]:
# --- setup: make `import ragkit` work from notebooks/ or solutions/ ---
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from ragkit.corpus import load_documents, load_corpus, load_qrels, tokenize
from ragkit.embed import get_embedder
from ragkit import llm

In [ ]:
# Which generation backend is active?
print("generation backend:", llm.available())

# Is the embedding model installed?
try:
    emb = get_embedder()
    print("embedder ready:", emb.model_name, "| dim =", emb.dim)
except ImportError as e:
    print("EMBEDDER NOT INSTALLED — run: pip install sentence-transformers\n")
    print(e)

## The corpus

Nine short Markdown docs for a fictional SaaS company, "Meridian". Each has
`##` sections (that structure matters in notebook 02). A small labelled
question set (`qrels`) lets us *measure* retrieval in notebook 05.


In [ ]:
docs = load_documents()
print(len(docs), "documents:\n", ", ".join(docs))

print("\n--- sample document: expense-policy ---\n")
print(docs["expense-policy"])

In [ ]:
qrels = load_qrels()
from collections import Counter
print("questions by kind:", dict(Counter(q["kind"] for q in qrels)), "\n")
for kind in ("lexical", "semantic", "multihop"):
    ex_q = next(q for q in qrels if q["kind"] == kind)
    print(f"[{kind:8}] {ex_q['question']}\n           gold -> {ex_q['gold_docs']}")

## The one idea behind dense retrieval

An **embedding model** maps text to a vector so that *similar meaning → nearby
vector*. "Retrieval" is then just: embed the query, find the nearest document
vectors. Let's see that directly.


In [ ]:
# (needs the embedder installed)
pairs = [
    ("How many holidays do I get?", "annual leave and vacation days"),   # related
    ("How many holidays do I get?", "the API returns HTTP 429 when throttled"),  # unrelated
]
for a, b in pairs:
    va, vb = get_embedder().encode(a), get_embedder().encode(b)
    print(f"cos = {float(va @ vb):+.3f}   {a!r}  vs  {b!r}")
# Rows are L2-normalised, so the dot product IS cosine similarity.

**Learning path**

| nb | idea |
|----|------|
| 01 | the minimal RAG loop: embed → cosine search → prompt → generate |
| 02 | chunking: why *how* you split decides what you can retrieve |
| 03 | hybrid search: BM25 (exact) + dense (meaning), fused with RRF |
| 04 | reranking: cheap recall, then a cross-encoder for precision |
| 05 | evaluation: recall@k and MRR — measure everything above |
| 06 | iterative RAG (advanced): multi-hop questions need >1 retrieval |

Do the exercises in each notebook (fill the `# YOUR CODE HERE` blanks; the
`assert`s check you). Solutions are in `solutions/`.
